In [ ]:
#FDA Load + NDC11 + Status

import requests
import pandas as pd
import time

url = "https://api.fda.gov/drug/ndc.json"

# Get total count
meta = requests.get(url).json()
total = meta["meta"]["results"]["total"]
print(f"Total records: {total}")

# Pull everything in pages of 1000
records = []
skip = 0
limit = 1000

while skip < total:
    params = {"limit": limit, "skip": skip}
    response = requests.get(url, params=params)
    data = response.json()
    
    for item in data.get("results", []):
        ingredients = item.get("active_ingredients") or []
        first_ingredient = ingredients[0] if ingredients else {}
        
        packaging = item.get("packaging") or []
        first_package = packaging[0] if packaging else {}
        
        flat = {
            "product_ndc": item.get("product_ndc"),
            "generic_name": item.get("generic_name"),
            "brand_name": item.get("brand_name"),
            "labeler_name": item.get("labeler_name"),
            "dosage_form": item.get("dosage_form"),
            "route": ", ".join(item.get("route") or []),
            "marketing_category": item.get("marketing_category"),
            "listing_expiration_date": item.get("listing_expiration_date"),
            "product_type": item.get("product_type"),
            "finished": item.get("finished"),
            "active_ingredient": first_ingredient.get("name"),
            "strength": first_ingredient.get("strength"),
            "package_ndc": first_package.get("package_ndc"),
            "rxcui": ", ".join(item.get("openfda", {}).get("rxcui") or []),
        }
        records.append(flat)
    
    skip += limit
    time.sleep(0.2)
    print(f"Pulled {min(skip, total)} / {total}")

df = pd.DataFrame(records)
print(f"\nFinal row count: {len(df)}")

def to_ndc11(ndc, hyphenated=False):
    if not ndc or not isinstance(ndc, str):
        return None
    ndc = ndc.strip()
    parts = ndc.split("-")
    if len(parts) == 1 and len(ndc) == 11 and ndc.isdigit():
        parts = [ndc[:5], ndc[5:9], ndc[9:]]
    elif len(parts) != 3 or not all(p.isdigit() for p in parts):
        return None
    lengths = tuple(len(p) for p in parts)
    if lengths not in {(4, 4, 2), (5, 3, 2), (5, 4, 1), (5, 4, 2)}:
        return None
    labeler = parts[0].zfill(5)
    product = parts[1].zfill(4)
    package = parts[2].zfill(2)
    if hyphenated:
        return f"{labeler}-{product}-{package}"
    return labeler + product + package

df["ndc11"] = df["package_ndc"].apply(lambda x: to_ndc11(x) if pd.notna(x) else None)
df["package_ndc_status"] = df["package_ndc"].apply(
    lambda x: "Available" if pd.notna(x) else "Package not available"
)

print(df[["package_ndc", "ndc11", "package_ndc_status"]].head(10))
print(f"Rows with valid ndc11: {df['ndc11'].notna().sum():,}")

In [ ]:
#Run to expand for dataset
pd.set_option("display.max_colwidth", 40)
df.head(1)

In [ ]:
# RxNorm NDC Crosswalk (from zip) — dynamic latest URL

import pandas as pd
import zipfile
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# Dynamically find the latest RxNorm full prescribe zip URL
page_url = "https://www.nlm.nih.gov/research/umls/rxnorm/docs/rxnormfiles.html"
resp = requests.get(page_url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

link = soup.find("a", href=lambda href: href and "RxNorm_full_prescribe_" in href and href.endswith(".zip"))
if not link:
    raise ValueError("Could not find latest RxNorm prescribe zip link on the page.")

url = link["href"]
zip_name = url.split("/")[-1]
zip_path = Path(zip_name)

print(f"Latest URL: {url}")

if not zip_path.exists():
    print("Downloading...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Unique NDCs:", ndc_xwalk["ndc11"].nunique())
print("Unique RxCUIs:", ndc_xwalk["rxcui"].nunique())

In [ ]:
#RxNav Concepts

import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

rxcuis = ndc_xwalk["rxcui"].dropna().unique().tolist()
print(f"Unique RxCUIs: {len(rxcuis):,}")

def fetch_concept(rxcui):
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            prop = data.get("properties", {})
            return {
                "rxcui": rxcui,
                "name": prop.get("name"),
                "tty": prop.get("tty"),
                "suppress": prop.get("suppress"),
            }
    except Exception:
        pass
    return {"rxcui": rxcui, "name": None, "tty": None, "suppress": None}

print("Fetching concepts from RxNav (parallel)...")
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_concept, rxcui): rxcui for rxcui in rxcuis}
    for i, future in enumerate(as_completed(futures), 1):
        results.append(future.result())
        if i % 500 == 0:
            print(f"  {i:,} / {len(rxcuis):,}")

concepts = pd.DataFrame(results)
print(f"Done. Concepts retrieved: {len(concepts):,}")
print(concepts.head())

In [ ]:
# Join All Three (corrected)
# FDA → Crosswalk on ndc11, then → Concepts on the crosswalk's rxcui

joined = (
    df.merge(ndc_xwalk, on="ndc11", how="left", suffixes=("", "_xwalk"))
      .merge(concepts, left_on="rxcui_xwalk", right_on="rxcui", how="left", suffixes=("", "_concept"))
)

print(f"Joined shape: {joined.shape}")
print(f"Matched to RxNorm crosswalk: {joined['rxcui_xwalk'].notna().sum():,}")
print(f"Matched to concept name: {joined['name'].notna().sum():,}")
joined.head(10)

In [ ]:
# TTY lookup
tty_lookup = pd.DataFrame({
    "tty": [
        "SCD", "SBD", "GPCK", "BPCK",
        "IN", "PIN", "BN", "DF",
        "SCDF", "SBDF", "SCDG", "SBDG",
        "MIN", "TMSY", "SY", "PSN"
    ],
    "term_type": [
        "Semantic Clinical Drug",
        "Semantic Branded Drug",
        "Generic Pack",
        "Branded Pack",
        "Ingredient",
        "Precise Ingredient",
        "Brand Name",
        "Dose Form",
        "Semantic Clinical Drug Form",
        "Semantic Branded Drug Form",
        "Semantic Clinical Drug Group",
        "Semantic Branded Drug Group",
        "Multiple Ingredients",
        "Tall Man Synonym",
        "Synonym",
        "Prescribable Name"
    ]
})

# Attach meaning to the joined data
joined = joined.merge(tty_lookup, on="tty", how="left")

print(f"TTY values mapped: {joined['term_type'].notna().sum():,}")
print("\nSample TTY breakdown:")
print(joined["term_type"].value_counts(dropna=False).head(15))

In [ ]:
#File Export

from pathlib import Path

out_folder = Path(r"C:\Add_Path_Here") 
out_folder.mkdir(parents=True, exist_ok=True)

# Final joined file
joined.to_parquet(out_folder / "fda_rxnorm_joined.parquet", index=False)
joined.to_csv(out_folder / "fda_rxnorm_joined.csv", index=False)

# Optional intermediate saves
ndc_xwalk.to_parquet(out_folder / "rxnorm_ndc_xwalk.parquet", index=False)
concepts.to_parquet(out_folder / "rxnav_concepts.parquet", index=False)

print("Exported.")